In [1]:
import pandas as pd
import os
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from tqdm import tqdm

In [2]:
if not os.path.exists('lenta-ru-news.csv.gz'):
    !wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

In [3]:
from corus import load_lenta

path = 'lenta-ru-news.csv.gz'
records = load_lenta(path)
next(records)

LentaRecord(
    url='https://lenta.ru/news/2018/12/14/cancer/',
    title='Названы регионы России с\xa0самой высокой смертностью от\xa0рака',
    text='Вице-премьер по социальным вопросам Татьяна Голикова рассказала, в каких регионах России зафиксирована наиболее высокая смертность от рака, сообщает РИА Новости. По словам Голиковой, чаще всего онкологические заболевания становились причиной смерти в Псковской, Тверской, Тульской и Орловской областях, а также в Севастополе. Вице-премьер напомнила, что главные факторы смертности в России — рак и болезни системы кровообращения. В начале года стало известно, что смертность от онкологических заболеваний среди россиян снизилась впервые за три года. По данным Росстата, в 2017 году от рака умерли 289 тысяч человек. Это на 3,5 процента меньше, чем годом ранее.',
    topic='Россия',
    tags='Общество',
    date=None
)

In [4]:
articles = []
for i in range(100000):
  record = (next(records))
  arcicle = {'title': record.title.lower(), 'topic': record.topic.lower(),
             'text': record.text.lower()}
  articles.append(arcicle)

# Чтобы не было ошибок при разделении на train, test, validation
topic_counts = Counter(article['topic'] for article in articles)
articles = [article for article in articles if topic_counts[article['topic']] >= 4]

In [5]:
df = pd.DataFrame(articles).drop_duplicates().reset_index(drop=True)

In [6]:
df.head()

,title,topic,text
0,австрия не представила доказательств вины росс...,спорт,австрийские правоохранительные органы не предс...
1,обнаружено самое счастливое место на планете,путешествия,сотрудники социальной сети instagram проанализ...
2,в сша раскрыли сумму расходов на расследование...,мир,с начала расследования российского вмешательст...
3,хакеры рассказали о планах великобритании зами...,мир,хакерская группировка anonymous опубликовала н...
4,архиепископ канонической упц отказался прийти ...,бывший ссср,архиепископ канонической украинской православн...


In [7]:
df.describe()

,title,topic,text
count,99996,99996,99996
unique,99847,18,99996
top,раскрыта стоимость самой дешевой съемной кварт...,россия,австрийские правоохранительные органы не предс...
freq,3,15150,1


In [8]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Semyon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
snowball = SnowballStemmer(language='russian')
stopWords = set(stopwords.words('russian'))

In [10]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Semyon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [11]:
def preprocess(text):
    tokens = [token for token in word_tokenize(text, language="russian")]
    tokens = [token for token in tokens if token not in stopWords]
    tokens = [snowball.stem(token) for token in tokens]

    return ' '.join(tokens)

In [12]:
df['preprocessed_text'] = [preprocess(text) for text in tqdm(df.title +
                           '/n' + df.text, total=len(df))]

100%|██████████| 99996/99996 [12:47<00:00, 130.31it/s]


In [13]:
df.head()

,title,topic,text,preprocessed_text
0,австрия не представила доказательств вины росс...,спорт,австрийские правоохранительные органы не предс...,австр представ доказательств вин российск биат...
1,обнаружено самое счастливое место на планете,путешествия,сотрудники социальной сети instagram проанализ...,обнаруж сам счастлив мест планете/нсотрудник с...
2,в сша раскрыли сумму расходов на расследование...,мир,с начала расследования российского вмешательст...,сша раскр сумм расход расследован « российск д...
3,хакеры рассказали о планах великобритании зами...,мир,хакерская группировка anonymous опубликовала н...,хакер рассказа план великобритан заминирова се...
4,архиепископ канонической упц отказался прийти ...,бывший ссср,архиепископ канонической украинской православн...,архиепископ каноническ упц отказа прийт « сата...


Пайплайн:
1.   Приводим заголовки и текст к нижнему регистру
2.   Объединяем заголовок с тектом, добавив между ними символ переноса строки
3. Токенизируем
4. Убираем из списка токенов стоп-слова
5. Используем стеммер

Не используется лемматизация, т.к. стемминг быстрее, а информация о форме слов не сильно скажется на качестве классификации, т.к. все тексты написаны журналистами.



In [14]:
X_train, x_tt, y_train, y_tt = train_test_split(df['preprocessed_text'], df.topic, test_size=0.4, random_state=42, stratify=df.topic)
X_val, X_test, y_val, y_test = train_test_split(x_tt, y_tt, test_size=0.5, random_state=42, stratify=y_tt)

# Baseline

In [15]:
dummyClassifier = DummyClassifier(strategy="most_frequent")
dummyClassifier.fit(X_train, y_train)
y_dummy_pred = dummyClassifier.predict(X_val)

In [16]:
print(classification_report(y_val, y_dummy_pred, zero_division=0))

                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.00      0.00      0.00       163
           бизнес       0.00      0.00      0.00       399
      бывший ссср       0.00      0.00      0.00      1362
              дом       0.00      0.00      0.00       681
         из жизни       0.00      0.00      0.00       980
   интернет и сми       0.00      0.00      0.00      1387
             крым       0.00      0.00      0.00       132
    культпросвет        0.00      0.00      0.00        62
         культура       0.00      0.00      0.00      1315
              мир       0.00      0.00      0.00      2884
  наука и техника       0.00      0.00      0.00      1129
      путешествия       0.00      0.00      0.00       645
           россия       0.15      1.00      0.26      3030
силовые структуры       0.00      0.00      0.00      1385
            спорт       0.00      0.00      0.00      2

# Логистическая регрессия с TfidfVectorizer

In [17]:
tfidfVectorizer = TfidfVectorizer()

In [18]:
log_reg = LogisticRegression(random_state=42, max_iter=1000)
pipelineTfidf = Pipeline([('vectorizer', tfidfVectorizer), ('classifier', log_reg)])
pipelineTfidf.fit(X_train, y_train)

,steps,"[('vectorizer', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [19]:
y_tfidf_pred = pipelineTfidf.predict(X_val)

In [20]:
print(classification_report(y_val, y_tfidf_pred, zero_division=0))

                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.94      0.60      0.73       163
           бизнес       0.71      0.41      0.52       399
      бывший ссср       0.87      0.86      0.86      1362
              дом       0.86      0.81      0.83       681
         из жизни       0.79      0.78      0.78       980
   интернет и сми       0.82      0.82      0.82      1387
             крым       0.76      0.58      0.66       132
    культпросвет        0.82      0.29      0.43        62
         культура       0.87      0.91      0.89      1315
              мир       0.84      0.90      0.87      2884
  наука и техника       0.87      0.90      0.89      1129
      путешествия       0.86      0.76      0.81       645
           россия       0.77      0.83      0.80      3030
силовые структуры       0.79      0.72      0.75      1385
            спорт       0.96      0.97      0.96      2

# Логистическая регрессия с CountVectorizer

In [21]:
countVectorizer = CountVectorizer()

In [22]:
pipelineCount = Pipeline([('vectorizer', countVectorizer), ('classifier', log_reg)])
pipelineCount.fit(X_train, y_train)

,steps,"[('vectorizer', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [23]:
y_count_pred = pipelineCount.predict(X_val)
print(classification_report(y_val, y_count_pred, zero_division=0))

                   precision    recall  f1-score   support

                        1.00      0.25      0.40         4
   69-я параллель       0.88      0.75      0.81       163
           бизнес       0.64      0.53      0.58       399
      бывший ссср       0.87      0.86      0.86      1362
              дом       0.85      0.85      0.85       681
         из жизни       0.79      0.79      0.79       980
   интернет и сми       0.82      0.83      0.82      1387
             крым       0.78      0.69      0.73       132
    культпросвет        0.63      0.47      0.54        62
         культура       0.87      0.89      0.88      1315
              мир       0.86      0.89      0.87      2884
  наука и техника       0.88      0.89      0.88      1129
      путешествия       0.85      0.81      0.83       645
           россия       0.78      0.82      0.80      3030
силовые структуры       0.77      0.74      0.76      1385
            спорт       0.97      0.97      0.97      2

# Подбор гиперпараметров

### TfidfVectorizer

In [24]:
parameters = {
    "vectorizer__min_df": [1, 3, 5],
    'vectorizer__max_df': (0.5, 0.75, 1.0),
    'classifier__C': (0.1, 1, 10),
}

grid_search_tfidf = GridSearchCV(pipelineTfidf, parameters, cv=3, scoring="accuracy")
grid_search_tfidf.fit(X_train, y_train)
print(f"Best parameters: {grid_search_tfidf.best_params_}")
print(f"Best score: {grid_search_tfidf.best_score_}")

Best parameters: {'classifier__C': 10, 'vectorizer__max_df': 0.5, 'vectorizer__min_df': 1}
Best score: 0.847492374618731


In [25]:
best_model_tfidf = grid_search_tfidf.best_estimator_
y_pred_test_tfidf = best_model_tfidf.predict(X_test)

In [26]:
print(classification_report(y_test, y_pred_test_tfidf, zero_division=0))

                   precision    recall  f1-score   support

                        0.00      0.00      0.00         3
   69-я параллель       0.95      0.74      0.83       163
           бизнес       0.67      0.52      0.58       398
      бывший ссср       0.87      0.88      0.87      1362
              дом       0.90      0.86      0.88       682
         из жизни       0.82      0.80      0.81       981
   интернет и сми       0.84      0.82      0.83      1387
             крым       0.80      0.70      0.74       132
    культпросвет        0.69      0.39      0.50        61
         культура       0.88      0.90      0.89      1316
              мир       0.86      0.89      0.87      2885
  наука и техника       0.89      0.90      0.90      1129
      путешествия       0.88      0.83      0.85       644
           россия       0.80      0.83      0.82      3030
силовые структуры       0.79      0.77      0.78      1385
            спорт       0.96      0.97      0.97      2

### CountVectorizer

In [27]:
grid_search_count = GridSearchCV(pipelineTfidf, parameters, cv=3, scoring="accuracy")
grid_search_count.fit(X_train, y_train)
print(f"Best parameters: {grid_search_count.best_params_}")
print(f"Best score: {grid_search_count.best_score_}")

Best parameters: {'classifier__C': 10, 'vectorizer__max_df': 0.5, 'vectorizer__min_df': 1}
Best score: 0.847492374618731


In [28]:
best_model_count = grid_search_count.best_estimator_
y_pred_test_count = best_model_count.predict(X_test)

In [29]:
print(classification_report(y_test, y_pred_test_count, zero_division=0))

                   precision    recall  f1-score   support

                        0.00      0.00      0.00         3
   69-я параллель       0.95      0.74      0.83       163
           бизнес       0.67      0.52      0.58       398
      бывший ссср       0.87      0.88      0.87      1362
              дом       0.90      0.86      0.88       682
         из жизни       0.82      0.80      0.81       981
   интернет и сми       0.84      0.82      0.83      1387
             крым       0.80      0.70      0.74       132
    культпросвет        0.69      0.39      0.50        61
         культура       0.88      0.90      0.89      1316
              мир       0.86      0.89      0.87      2885
  наука и техника       0.89      0.90      0.90      1129
      путешествия       0.88      0.83      0.85       644
           россия       0.80      0.83      0.82      3030
силовые структуры       0.79      0.77      0.78      1385
            спорт       0.96      0.97      0.97      2